# Semantic caching for LISA with RedisVL `SemanticCache`

Based on the RedisVL 0.4.1 user guide **"Semantic Caching for LLMs"** (https://docs.redisvl.com/en/0.4.1/user_guide/03_llmcache.html), adapted to LISA:

| Tutorial | This notebook |
|---|---|
| OpenAI `gpt-3.5-turbo-instruct` | **Azure OpenAI** from LISA's `.env` (`AZURE_LLM_*`), or OpenAI, or an offline stand-in |
| Hugging Face `all-mpnet-base-v2` embeddings (the default) | **Ollama `qwen3-embedding:0.6b`**, the model LISA already uses (Hugging Face is still an option) |
| `!rvl index info` / `!rvl stats` | The same information from Python, so it works on Windows without the `rvl` command on PATH |
| — | **Part 2: LISA.** Why a semantic cache must not answer tickets, how to pick a safe threshold from data, and a safe FAQ pattern |

A **semantic cache** stores `prompt -> response`. When a new prompt is *close enough* in meaning to a stored one (vector distance below a threshold), it returns the stored response and does **not** call the LLM.
This is different from LISA's existing caches, which only reuse an answer for an **identical** input.

## 0. Before you run it (Windows)

**Redis with vector search.** `SemanticCache` needs the Redis Query Engine (RediSearch).
- Use **Redis Stack** or **Redis 8**: `docker run -d --name redis-stack -p 6379:6379 redis/redis-stack-server:latest`
- **Memurai and plain Redis 7 will not work:** they fail with `unknown command 'FT.CREATE'`.
- If LISA already uses plain Redis on 6379, start Redis Stack on another port (`-p 6380:6379`) and set `SEMCACHE_REDIS_URL=redis://localhost:6380` below.

**Embeddings.**
- Ollama running with `ollama pull qwen3-embedding:0.6b` (LISA already needs it).
- Or set `EMBEDDINGS = "hf"` to use the tutorial's Hugging Face model. It downloads about 420 MB the first time.

**Start Jupyter from the LISA folder.** `--with` adds the packages for this run only; `pyproject.toml` is not changed:
```powershell
uv run --with "redisvl==0.4.1" --with python-dotenv jupyter lab
```

In [ ]:
# Only if you are NOT using the uv command above:
# %pip install "redisvl==0.4.1" python-dotenv openai ollama

## 1. Settings

In [ ]:
import os
import time
from pathlib import Path

import numpy as np

try:                                   # LISA's .env (AZURE_LLM_*, OLLAMA_BASE_URL); the notebook also works without it
    from dotenv import load_dotenv
    load_dotenv(os.getenv("LISA_ENV_FILE", ".env"))
except ImportError:
    pass

os.environ["TOKENIZERS_PARALLELISM"] = "false"

REDIS_URL = os.getenv("SEMCACHE_REDIS_URL", "redis://localhost:6379")      # must be Redis Stack / Redis 8 (see section 0)
EMBEDDINGS = os.getenv("SEMCACHE_EMBEDDINGS", "ollama")                    # "ollama" (LISA's model) or "hf" (the tutorial's model)
OLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_EMBED_MODEL = os.getenv("SEMCACHE_OLLAMA_MODEL", "qwen3-embedding:0.6b")
LLM = os.getenv("SEMCACHE_LLM", "auto")                                    # "auto" = Azure if AZURE_LLM_API_KEY is set, else OpenAI if OPENAI_API_KEY is set, else offline

print(f"Redis: {REDIS_URL} | embeddings: {EMBEDDINGS} | LLM: {LLM}")

## 2. The LLM: an `ask_llm` helper

The tutorial calls OpenAI's legacy completions model. Here the helper calls the chat model LISA is configured with.
The Azure endpoint is built the same way as in LISA's `workflows/common.py`.
**Offline** is a stand-in for running the notebook without a key: it waits 1.5 s, like a real call, and returns a fixed text.

In [ ]:
def _make_llm():
    use = LLM
    if use == "auto":
        use = "azure" if os.getenv("AZURE_LLM_API_KEY") else "openai" if os.getenv("OPENAI_API_KEY") else "offline"
    if use == "azure":
        from openai import AzureOpenAI
        client = AzureOpenAI(
            azure_endpoint=os.environ["AZURE_LLM_BASE_URL"].removesuffix("/openai"),
            api_key=os.environ["AZURE_LLM_API_KEY"],
            api_version=os.environ["AZURE_LLM_API_VERSION"],
            timeout=float(os.getenv("AZURE_LLM_TIMEOUT", "60")),
        )
        model = os.environ["AZURE_LLM_DEPLOYMENT"]
    elif use == "openai":
        from openai import OpenAI
        client, model = OpenAI(), os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    else:
        def offline(question: str) -> str:
            time.sleep(1.5)
            return f"(offline stand-in answer to: {question})"
        return "offline", offline

    def ask(question: str) -> str:
        response = client.chat.completions.create(
            model=model, messages=[{"role": "user", "content": question}], max_tokens=200, temperature=0,
        )
        return response.choices[0].message.content.strip()
    return use, ask


LLM_USED, ask_llm = _make_llm()
print("LLM:", LLM_USED)
print(ask_llm("What is the capital of France? Answer in one word."))

## 3. The embedding model (vectorizer)

`SemanticCache` turns every prompt into a vector with a **vectorizer**.
- **Default (tutorial):** `HFTextVectorizer("sentence-transformers/all-mpnet-base-v2")`.
- **This notebook:** LISA's Ollama model, wrapped in RedisVL's `CustomTextVectorizer`.

A cache compares a **prompt with a prompt**, so no query instruction prefix is added (LISA's KB search adds one because it compares a query with a document).

> **Distance thresholds depend on the model.** The tutorial's `0.1` and `0.3` were chosen for `all-mpnet-base-v2`. With another model, measure first (section 6 and Part 2).

In [ ]:
from redisvl.utils.vectorize import CustomTextVectorizer, HFTextVectorizer

if EMBEDDINGS == "hf":
    vectorizer = HFTextVectorizer(model="sentence-transformers/all-mpnet-base-v2")
else:
    import ollama

    _ollama = ollama.Client(host=OLLAMA_URL)

    def _embed_many(texts, **kwargs):
        return [list(map(float, v)) for v in _ollama.embed(model=OLLAMA_EMBED_MODEL, input=list(texts))["embeddings"]]

    def _embed(text, **kwargs):
        return _embed_many([text])[0]

    vectorizer = CustomTextVectorizer(embed=_embed, embed_many=_embed_many)

print(type(vectorizer).__name__, "| dimensions:", vectorizer.dims)

## 4. Initializing `SemanticCache`

`SemanticCache` automatically creates a search index in Redis for the cache content.
`overwrite=True` rebuilds the index if it exists with another schema, for example after switching embedding models (the dimensions differ).

In [ ]:
from redisvl.extensions.llmcache import SemanticCache

llmcache = SemanticCache(
    name="llmcache",                 # underlying search index name
    redis_url=REDIS_URL,             # redis connection url string
    distance_threshold=0.1,          # semantic cache distance threshold
    vectorizer=vectorizer,
    overwrite=True,
)

In [ ]:
# Look at the index specification created for the semantic cache (what `rvl index info -i llmcache` prints)
info = llmcache.index.info()
print("index:", info["index_name"], "| documents:", info["num_docs"])
for attribute in info["attributes"]:
    a = dict(zip(attribute[::2], attribute[1::2]))
    print(f"  {a.get('identifier')!s:22} {a.get('type')}")

## 5. Basic cache usage

`check()` returns a list of hits, which is empty on a miss. The helper below prints the response **and the vector distance**: you need the distance to choose a threshold.
(The tutorial indexes `check(...)[0]`, which raises `IndexError` on a miss.)

In [ ]:
def show(results):
    if not results:
        print("MISS (nothing within the threshold)")
    for r in results:
        print(f"HIT  distance={float(r['vector_distance']):.4f}  prompt={r['prompt']!r}  ->  {r['response']!r}")


question = "What is the capital of France?"
show(llmcache.check(prompt=question))          # empty cache -> MISS

Store the question, the response, and any metadata (a Python dict):

In [ ]:
llmcache.store(
    prompt=question,
    response="Paris",
    metadata={"city": "Paris", "country": "france"},
)

In [ ]:
# The same question again: a hit, with the metadata
print(llmcache.check(prompt=question, return_fields=["prompt", "response", "metadata"]))

In [ ]:
# A semantically similar question
show(llmcache.check(prompt="What actually is the capital of France?"))

## 6. Customize the distance threshold

The right threshold depends on the embedding model, the kind of question and the business risk. It can be changed at any time with `set_threshold`.
Use `check(distance_threshold=...)` to try one value for one call only.

In [ ]:
tricky = "What is the capital city of the country in Europe that also has a city named Nice?"
show(llmcache.check(prompt=tricky))                              # threshold 0.1
show(llmcache.check(prompt=tricky, distance_threshold=1.0))      # just to SEE its distance

In [ ]:
llmcache.set_threshold(0.3)          # widen the threshold
show(llmcache.check(prompt=tricky))

In [ ]:
# Invalidate the cache completely by clearing it out (the index stays)
llmcache.clear()
show(llmcache.check(prompt=tricky))

## 7. TTL: entries that expire on their own

A TTL on the cache applies to every new entry. `store(..., ttl=...)` sets one entry's TTL. `set_ttl()` with no argument removes the cache-wide TTL.

In [ ]:
llmcache.set_ttl(5)                                           # 5 seconds
llmcache.store("This is a TTL test", "This is a TTL test response")
time.sleep(6)
print(llmcache.check("This is a TTL test"))                  # [] - expired
llmcache.set_ttl()                                            # back to no TTL

## 8. Simple performance test

Compare one LLM call with the average of 10 cache hits.

In [ ]:
def answer_question(question: str) -> str:
    # Check the semantic cache first, call the LLM only on a miss
    results = llmcache.check(prompt=question)
    if results:
        return results[0]["response"]
    return ask_llm(question)


question = "What was the name of the first US President?"
start = time.time()
answer = answer_question(question)
end = time.time()
print(f"Without caching ({LLM_USED}) the answer took {end - start:.2f} s: {answer!r}")

llmcache.store(prompt=question, response="George Washington")

In [ ]:
times = []
for _ in range(10):
    t0 = time.time()
    answer_question(question)
    times.append(time.time() - t0)

avg_time_with_cache = float(np.mean(times))
print(f"Average time with the cache: {avg_time_with_cache:.4f} s")
print(f"Time saved: {round(((end - start) - avg_time_with_cache) / (end - start) * 100, 2)}%")
print("Note: a cache hit still costs one embedding call; that is most of the remaining time.")

In [ ]:
# Index stats (what `rvl stats -i llmcache` prints)
info = llmcache.index.info()
print({k: info[k] for k in ("num_docs", "num_records", "vector_index_sz_mb", "total_indexing_time") if k in info})

In [ ]:
# Clear the cache AND delete the underlying index
llmcache.delete()

## 9. Access controls, tags and filters

Several users or applications can share one cache **only if** their entries are kept apart. `filterable_fields` adds tag or numeric fields, and `check(filter_expression=...)` returns only matching entries.

In [ ]:
from redisvl.query.filter import Num, Tag

private_cache = SemanticCache(
    name="private_cache",
    redis_url=REDIS_URL,
    vectorizer=vectorizer,
    filterable_fields=[{"name": "user_id", "type": "tag"}],
    overwrite=True,
)

private_cache.store(
    prompt="What is the phone number linked to my account?",
    response="The number on file is 123-555-0000",
    filters={"user_id": "abc"},
)
private_cache.store(
    prompt="What's the phone number linked in my account?",
    response="The number on file is 123-555-1111",
    filters={"user_id": "def"},
)

response = private_cache.check(
    prompt="What is the phone number linked to my account?",
    filter_expression=Tag("user_id") == "abc",
    num_results=2,
)
print(f"found {len(response)} entry")
show(response)
private_cache.delete()

Several filterable fields can be combined, together with the default fields:

In [ ]:
complex_cache = SemanticCache(
    name="account_data",
    redis_url=REDIS_URL,
    vectorizer=vectorizer,
    filterable_fields=[
        {"name": "user_id", "type": "tag"},
        {"name": "account_type", "type": "tag"},
        {"name": "account_balance", "type": "numeric"},
        {"name": "transaction_amount", "type": "numeric"},
    ],
    overwrite=True,
)
complex_cache.store(prompt="what is my most recent checking account transaction under $100?", response="Your most recent transaction was for $75",
                    filters={"user_id": "abc", "account_type": "checking", "transaction_amount": 75})
complex_cache.store(prompt="what is my most recent savings account transaction?", response="Your most recent deposit was for $300",
                    filters={"user_id": "abc", "account_type": "savings", "transaction_amount": 300})
complex_cache.store(prompt="what is my most recent checking account transaction over $200?", response="Your most recent transaction was for $350",
                    filters={"user_id": "abc", "account_type": "checking", "transaction_amount": 350})
complex_cache.store(prompt="what is my checking account balance?", response="Your current checking account is $1850",
                    filters={"user_id": "abc", "account_type": "checking"})

complex_filter = (Num("transaction_amount") > 100) & (Tag("account_type") == "checking")
complex_cache.set_threshold(0.3)
response = complex_cache.check(
    prompt="what is my most recent checking account transaction?",
    filter_expression=complex_filter,
    num_results=5,
)
print(f"found {len(response)} entry")
show(response)
complex_cache.delete()

---
# Part 2: semantic caching and LISA

LISA's caching plan **rejects a semantic cache for ticket answers**. Two tickets that read almost the same can need different answers:
- a different device (laptop or phone);
- a different CI (configuration item);
- a different reporter (the contact number, the staff number).

A wrong cached answer would be sent to a real person. Section 10 shows the failure, section 11 measures a safe threshold, and section 12 shows the one place where a semantic cache is reasonable.

## 10. The failure: near-duplicate tickets

In [ ]:
demo = SemanticCache(name="lisa_ticket_demo", redis_url=REDIS_URL, vectorizer=vectorizer, distance_threshold=0.3, overwrite=True)

demo.store(
    prompt="VPN is not connecting on my laptop, it says authentication failed",
    response="Laptop: open GlobalProtect > Settings > sign out, sign in again with your staff ID ... (KB0010001)",
)

for ticket in [
    "VPN is not connecting on my laptop, it says authentication failed",      # identical -> correct hit
    "VPN not connecting on my laptop, authentication failed error",           # same problem, other words -> a hit is fine
    "VPN is not connecting on my iPhone, it says authentication failed",      # OTHER DEVICE -> a hit is WRONG
    "Outlook is not connecting on my laptop, it says authentication failed",  # OTHER APPLICATION -> a hit is WRONG
]:
    print(ticket)
    show(demo.check(prompt=ticket, distance_threshold=1.0))   # show the distance even when it is above the threshold
    print()
demo.delete()

The **wrong** prompts usually sit only a little further away than the **right** ones; check the distances printed above.
With a loose threshold they are served the laptop VPN answer. Thresholds cannot reliably separate "same problem" from "same words, different problem".
For triage this is why LISA reuses an LLM answer **only for an identical prompt** (`LLM_CACHE_BACKEND`), and caches embeddings and reranker scores rather than answers.

## 11. Choosing a threshold from data, not by guessing

Label pairs as **should hit** (same question, the same answer is correct) or **must miss** (similar wording, different answer).
Then pick the **largest threshold with zero wrong hits**.
Use real questions for this: around 50 pairs from your FAQ or chat logs is far better than the handful below.

In [ ]:
pairs = [
    # (stored prompt, new prompt, should_hit)
    ("How do I reset my password?", "How can I reset my password?", True),
    ("How do I reset my password?", "I forgot my password, how do I change it?", True),
    ("How do I reset my password?", "How do I reset my MFA / authenticator app?", False),
    ("How do I clear the Chrome cache?", "How to clear cache in Google Chrome", True),
    ("How do I clear the Chrome cache?", "How do I clear the Edge cache?", False),
    ("How do I connect to the VPN from home?", "How to connect VPN when working from home", True),
    ("How do I connect to the VPN from home?", "How do I connect to the office Wi-Fi?", False),
    ("How do I map a network drive?", "How to map a shared drive in Windows", True),
    ("How do I map a network drive?", "How do I add a network printer?", False),
]

stored = sorted({a for a, _, _ in pairs})
new = [b for _, b, _ in pairs]
vectors = dict(zip(stored + new, vectorizer.embed_many(stored + new)))


def cosine_distance(a, b):            # the distance RedisVL reports: 1 - cosine similarity
    a, b = np.asarray(a), np.asarray(b)
    return float(1 - a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


rows = [(a, b, hit, cosine_distance(vectors[a], vectors[b])) for a, b, hit in pairs]
for a, b, hit, d in sorted(rows, key=lambda r: r[3]):
    print(f"{d:.4f}  {'should hit' if hit else 'MUST MISS '}  {b!r}  vs  {a!r}")

In [ ]:
should_hit = sum(1 for r in rows if r[2])
print("threshold | correct hits | WRONG hits")
best, best_good = None, 0
for t in np.round(np.arange(0.02, 0.42, 0.02), 2):
    good = sum(1 for *_, hit, d in rows if hit and d <= t)
    bad = sum(1 for *_, hit, d in rows if not hit and d <= t)
    print(f"   {t:.2f}   |     {good}/{should_hit}      |    {bad}")
    if bad == 0:
        best, best_good = t, good

if best is None or best_good == 0:
    print("\nNo threshold gives a correct hit without a wrong one: with this embedding model a semantic cache is NOT safe for these questions.")
else:
    print(f"\nLargest threshold with no wrong hit: {best} -> {best_good}/{should_hit} correct hits. Use a margin below it in production.")

## 12. The safe pattern for LISA: a scoped FAQ cache

A semantic cache is reasonable only for **generic how-to answers** that do not depend on the ticket or the person, for example "how do I clear the Chrome cache".
Rules this pattern follows:

1. **Scope with filters.** A hit must have the same topic (`ci`, or a playbook) **and** the same `cache_version`. Bump the version when the KB or the prompts change.
2. **Never cache personal or ticket data.** No reporter name, staff number, phone number or ticket number in the prompt or the response.
3. **A tight threshold** from section 11, and a **TTL** so old answers age out.
4. **Store only good answers.** For example, only after the reporter confirmed it fixed the problem; LISA's `outcomes` table records exactly that.

In [ ]:
import re

CACHE_VERSION = "kb-2026-10"          # bump when the KB / prompts change -> old entries are never matched
FAQ_THRESHOLD = 0.1                   # take it from section 11 (with a margin), for YOUR embedding model

faq_cache = SemanticCache(
    name="lisa_faq",
    redis_url=REDIS_URL,
    vectorizer=vectorizer,
    distance_threshold=FAQ_THRESHOLD,
    ttl=24 * 3600,
    filterable_fields=[{"name": "ci", "type": "tag"}, {"name": "cache_version", "type": "tag"}],
    overwrite=True,
)

_PERSONAL = re.compile(r"\b(INC\d{5,}|S\d{5,}|\+?\d[\d\s-]{7,}\d|[\w.+-]+@[\w-]+\.[\w.]+)\b", re.I)


def answer_faq(question: str, ci: str, confirmed_good: bool | None = None) -> tuple[str, str]:
    # Returns (answer, "cache" | "llm"). Personal / ticket data is never looked up or stored.
    if _PERSONAL.search(question):
        return ask_llm(question), "llm (personal data: not cached)"
    scope = (Tag("ci") == ci) & (Tag("cache_version") == CACHE_VERSION)
    hits = faq_cache.check(prompt=question, filter_expression=scope)
    if hits:
        return hits[0]["response"], f"cache (distance {float(hits[0]['vector_distance']):.4f})"
    answer = ask_llm(question)
    if confirmed_good:                                    # e.g. the reporter said it fixed it (LISA's outcomes.reporter_confirmed)
        faq_cache.store(prompt=question, response=answer, filters={"ci": ci, "cache_version": CACHE_VERSION})
    return answer, "llm"


print(answer_faq("How do I clear the Chrome cache?", ci="CHROME", confirmed_good=True))
print(answer_faq("How do I clear the Chrome cache?", ci="CHROME"))                 # same scope -> cache
print(answer_faq("How do I clear the Chrome cache?", ci="EDGE"))                   # other CI -> never served the Chrome answer
print(answer_faq("My number is +971 50 123 4567, how do I clear the Chrome cache?", ci="CHROME"))

In [ ]:
# Cleanup
faq_cache.delete()

## 13. Summary for LISA

| Question | Answer |
|---|---|
| Use a semantic cache for ticket triage, classification or replies to reporters? | **No.** Similar wording does not mean the same answer (section 10). LISA keeps exact-match LLM caching. |
| Where can it help? | Generic how-to / FAQ answers, scoped by topic and version, with a measured threshold (sections 11 and 12) |
| What it needs | Redis Stack or Redis 8 (not Memurai or plain Redis 7), plus one embedding call per lookup |
| Threshold | Measure it per embedding model on labeled pairs. The tutorial's 0.1 / 0.3 are for `all-mpnet-base-v2` only. |
| Invalidate | Change `CACHE_VERSION` (filter), `clear()`, TTL, or `drop(ids=[...])` for one entry |